In [ ]:
import kaggle_benchmarks as kbench
print(kbench.llms.keys())


In [ ]:
!apt-get update && apt-get install -y git

In [ ]:
!git clone https://github.com/Gl12321/web_parse_agent

In [ ]:
!pip install -r web_parse_agent/requirements.txt

In [ ]:
!playwright install chromium --with-deps

In [ ]:
!pip uninstall -y openai
!pip install "openai<2.0.0"

In [ ]:
import kaggle_benchmarks as kbench
from kaggle_benchmarks.messages import Message
from kaggle_benchmarks import actors
import inspect
from typing import Any, List, Optional
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, SystemMessage, HumanMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.callbacks.manager import CallbackManagerForLLMRun

class KBenchLLMWrapper(BaseChatModel):
    kbench_llm: Any

    @property
    def _llm_type(self) -> str:
        return "kbench_llm"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        system_content = None
        kbench_messages = []

        for m in messages:
            if isinstance(m, SystemMessage):
                system_content = m.content
            elif isinstance(m, HumanMessage):
                kbench_messages.append(Message(sender=actors.user, content=m.content))
            elif isinstance(m, AIMessage):
                kbench_messages.append(Message(sender=actors.assistant, content=m.content))

        sig = inspect.signature(self.kbench_llm.invoke)
        if 'system' in sig.parameters:
            response = self.kbench_llm.invoke(messages=kbench_messages, system=system_content)
        else:
            final_messages = [SystemMessage(content=system_content)] + messages if system_content else messages
            response = self.kbench_llm.invoke(final_messages)

        content = getattr(response, 'content', str(response))
        if isinstance(response, dict):
            content = response.get('content') or response.get('text') or str(response)

        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=content))])

kbench_model = kbench.llms["openai/gpt-5.4-mini-2026-03-17"]
llm = KBenchLLMWrapper(kbench_llm=kbench_model)

In [ ]:
import os
import sys
import json
from typing import Optional, Type
from pydantic import BaseModel

if '/kaggle/working/web_parse_agent' not in sys.path:
    sys.path.insert(0, '/kaggle/working/web_parse_agent')

from src.infrastructure.crawler import Crawl4AIAdapter
from src.agent.agents.extractor_agent import ExtractorAgent
from src.agent.agents.navigator_agent import NavigatorAgent
from src.agent.agents.aggregator_agent import AggregatorAgent
from src.agent.agents.crawler_agent import CrawlerAgent
from src.agent.graph import WebParsingGraph
from src.extractors.schemas.contact import ContactInfo


def run_parser(
        llm, 
        url: str,
        goal: str,
        mode: str = "flexible",
        schema: Optional[Type[BaseModel]] = None,
        max_depth: int = 2,
        max_pages: int = 5
) -> dict:

    navigator = NavigatorAgent(llm)
    extractor = ExtractorAgent(llm)
    aggregator = AggregatorAgent(llm)
    crawler = CrawlerAgent()

    graph = WebParsingGraph(
        navigator=navigator,
        extractor=extractor,
        aggregator=aggregator,
        crawler=crawler,
        max_depth=max_depth,
        max_pages=max_pages
    )

    print(f"Starting crawl: {url}")
    print(f"Goal: {goal}")
    print(f"Mode: {mode}, Max depth: {max_depth}, Max pages: {max_pages}")
    print("-" * 50)

    result = graph.run(
        start_url=url,
        goal=goal,
        mode=mode,
        schema=schema
    )

    print("-" * 50)
    print(f"Pages processed: {result['pages_processed']}")
    print(f"Visited: {result['visited_urls']}")

    return result

def main(llm):
    print("EXAMPLE 1: Flexible extraction")

    url = "https://jet.su/"

    result = run_parser(
        llm=llm,
        url=url,
        goal="Extract strictly only minimal key information about company",
        mode="flexible",
        max_depth=3,
        max_pages=5
    )

    print("\nFinal result:")
    import json
    print(json.dumps(result['final_result'], indent=2, ensure_ascii=False))

    print("EXAMPLE 2: Strict extraction with Pydantic schema")

    result = run_parser(
        llm=llm,
        url=url,
        goal="Extract key contact info",
        mode="strict",
        schema=ContactInfo,
        max_depth=3,
        max_pages=5
    )

    print("\nFinal result:")
    print(json.dumps(result['final_result'], indent=2, ensure_ascii=False))

main(llm)